In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sqlalchemy import create_engine
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.drawing.image import Image as XLImage

# CONNECTION
SERVER   = "localhost\\SQLEXPRESS"
DATABASE = "retail_sales"
engine   = create_engine(
    f"mssql+pyodbc://{SERVER}/{DATABASE}"
    f"?driver=ODBC+Driver+17+for+SQL+Server"
    f"&trusted_connection=yes"
)

# LOAD TABLES
transactions     = pd.read_sql("SELECT * FROM transactions", engine)
customer_summary = pd.read_sql("SELECT * FROM customer_summary", engine)
monthly_summary  = pd.read_sql("SELECT * FROM monthly_summary", engine)
response         = pd.read_sql("SELECT * FROM customer_response", engine)

transactions["trans_date"]  = pd.to_datetime(transactions["trans_date"])
transactions["trans_month"] = transactions["trans_date"].dt.month
transactions["trans_year"]  = transactions["trans_date"].dt.year

# Merge customer summary with response
merged = customer_summary.merge(response, on="customer_id", how="inner")

print("Data loaded successfully.")
print(f"  transactions     : {len(transactions):,} rows")
print(f"  customer_summary : {len(customer_summary):,} rows")
print(f"  monthly_summary  : {len(monthly_summary):,} rows")
print(f"  response         : {len(response):,} rows")

Data loaded successfully.
  transactions     : 125,000 rows
  customer_summary : 6,889 rows
  monthly_summary  : 47 rows
  response         : 6,884 rows


In [2]:
# CELL 2: BUILD EXCEL REPORT

OUTPUT_PATH = r"C:\Users\Asus\Downloads\archive\Retail_Sales_Report.xlsx"
CHART_PATH  = r"C:\Users\Asus\Downloads\archive\\"

wb = Workbook()

# -- HELPER STYLES --
def style_header(cell, bg_color="1F4E79"):
    cell.font      = Font(bold=True, color="FFFFFF", size=11)
    cell.fill      = PatternFill("solid", fgColor=bg_color)
    cell.alignment = Alignment(horizontal="center", vertical="center")

def style_subheader(cell):
    cell.font      = Font(bold=True, color="FFFFFF", size=10)
    cell.fill      = PatternFill("solid", fgColor="2E75B6")
    cell.alignment = Alignment(horizontal="center", vertical="center")

def style_title(cell):
    cell.font      = Font(bold=True, size=14, color="1F4E79")
    cell.alignment = Alignment(horizontal="left", vertical="center")

def add_border(ws, min_row, max_row, min_col, max_col):
    thin = Side(style="thin")
    for row in ws.iter_rows(min_row=min_row, max_row=max_row,
                            min_col=min_col, max_col=max_col):
        for cell in row:
            cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

print("Workbook created. Building sheets...")

Workbook created. Building sheets...


In [4]:
# CELL 3: SUMMARY SHEET

ws_summary = wb.active
ws_summary.title = "Executive Summary"

# Title
ws_summary.merge_cells("A1:F1")
title_cell = ws_summary["A1"]
title_cell.value = "Retail Sales Analysis - Executive Summary"
style_title(title_cell)
ws_summary.row_dimensions[1].height = 30

# Key Metrics Section
ws_summary.merge_cells("A3:F3")
ws_summary["A3"].value = "Key Metrics"
style_header(ws_summary["A3"])
ws_summary.row_dimensions[3].height = 20

total_revenue    = transactions["tran_amount"].sum()
avg_transaction  = transactions["tran_amount"].mean()
avg_spend        = customer_summary["total_spend"].mean()
response_rate    = response["response"].sum() / len(response) * 100
unique_customers = customer_summary["customer_id"].nunique()

metrics = [
    ("Total Transactions",      f"{len(transactions):,}"),
    ("Total Revenue",           f"${total_revenue:,.2f}"),
    ("Unique Customers",        f"{unique_customers:,}"),
    ("Avg Transaction Value",   f"${avg_transaction:.2f}"),
    ("Avg Customer Spend",      f"${avg_spend:.2f}"),
    ("Promotion Response Rate", f"{response_rate:.2f}%"),
    ("Date Range",              "May 2011 - March 2015"),
    ("Total Months",            f"{len(monthly_summary)}"),
]

headers = ["Metric", "Value"]
for col, h in enumerate(headers, start=1):
    cell = ws_summary.cell(row=4, column=col, value=h)
    style_subheader(cell)

for i, (metric, value) in enumerate(metrics, start=5):
    ws_summary.cell(row=i, column=1, value=metric)
    ws_summary.cell(row=i, column=2, value=value)
    if i % 2 == 0:
        for col in [1, 2]:
            ws_summary.cell(row=i, column=col).fill = PatternFill("solid", fgColor="DEEAF1")

add_border(ws_summary, 4, 12, 1, 2)

ws_summary.column_dimensions["A"].width = 30
ws_summary.column_dimensions["B"].width = 25

print("Executive Summary sheet created.")

Executive Summary sheet created.


In [5]:
# CELL 4: MONTHLY SALES SHEET

ws_monthly = wb.create_sheet("Monthly Sales")

# Title
ws_monthly.merge_cells("A1:F1")
ws_monthly["A1"].value = "Monthly Sales Summary"
style_title(ws_monthly["A1"])
ws_monthly.row_dimensions[1].height = 30

# Add period column
monthly_summary["period"] = (
    monthly_summary["trans_year"].astype(str) + "-" +
    monthly_summary["trans_month"].astype(str).str.zfill(2)
)

monthly_report = monthly_summary[["period", "trans_year", "trans_month",
                                   "total_sales", "total_orders", "avg_order_value"]].copy()
monthly_report.columns = ["Period", "Year", "Month", "Total Sales ($)",
                           "Total Orders", "Avg Order Value ($)"]

# Headers
for col, header in enumerate(monthly_report.columns, start=1):
    cell = ws_monthly.cell(row=3, column=col, value=header)
    style_subheader(cell)

# Data
for row_idx, row in enumerate(monthly_report.itertuples(index=False), start=4):
    for col_idx, value in enumerate(row, start=1):
        cell = ws_monthly.cell(row=row_idx, column=col_idx, value=value)
        cell.alignment = Alignment(horizontal="center")
    if row_idx % 2 == 0:
        for col_idx in range(1, len(monthly_report.columns) + 1):
            ws_monthly.cell(row=row_idx, column=col_idx).fill = PatternFill("solid", fgColor="DEEAF1")

add_border(ws_monthly, 3, 3 + len(monthly_report), 1, len(monthly_report.columns))

for col in ["A", "B", "C", "D", "E", "F"]:
    ws_monthly.column_dimensions[col].width = 20

print("Monthly Sales sheet created.")


Monthly Sales sheet created.


In [6]:
# CELL 5: CUSTOMER SUMMARY SHEET

ws_customers = wb.create_sheet("Customer Summary")

# Title
ws_customers.merge_cells("A1:G1")
ws_customers["A1"].value = "Customer Summary Report"
style_title(ws_customers["A1"])
ws_customers.row_dimensions[1].height = 30

# Top 20 customers by spend
top20 = customer_summary.nlargest(20, "total_spend")[
    ["customer_id", "total_spend", "avg_transaction",
     "transaction_count", "first_purchase", "last_purchase", "lifespan_days"]
].copy()
top20.columns = ["Customer ID", "Total Spend ($)", "Avg Transaction ($)",
                 "Transaction Count", "First Purchase", "Last Purchase", "Lifespan (Days)"]

# Section header
ws_customers.merge_cells("A3:G3")
ws_customers["A3"].value = "Top 20 Customers by Total Spend"
style_header(ws_customers["A3"])

# Column headers
for col, header in enumerate(top20.columns, start=1):
    cell = ws_customers.cell(row=4, column=col, value=header)
    style_subheader(cell)

# Data
for row_idx, row in enumerate(top20.itertuples(index=False), start=5):
    for col_idx, value in enumerate(row, start=1):
        cell = ws_customers.cell(row=row_idx, column=col_idx, value=str(value))
        cell.alignment = Alignment(horizontal="center")
    if row_idx % 2 == 0:
        for col_idx in range(1, len(top20.columns) + 1):
            ws_customers.cell(row=row_idx, column=col_idx).fill = PatternFill("solid", fgColor="DEEAF1")

add_border(ws_customers, 4, 4 + len(top20), 1, len(top20.columns))

# Response summary section
ws_customers["A27"].value = "Customer Response Summary"
style_header(ws_customers["A27"])
ws_customers.merge_cells("A27:G27")

response_summary = merged.groupby("response")[
    ["total_spend", "avg_transaction", "transaction_count"]
].mean().round(2).reset_index()
response_summary["response"] = response_summary["response"].map({0: "Did Not Respond", 1: "Responded"})
response_summary.columns     = ["Response Group", "Avg Total Spend ($)",
                                 "Avg Transaction ($)", "Avg Transaction Count"]

for col, header in enumerate(response_summary.columns, start=1):
    cell = ws_customers.cell(row=28, column=col, value=header)
    style_subheader(cell)

for row_idx, row in enumerate(response_summary.itertuples(index=False), start=29):
    for col_idx, value in enumerate(row, start=1):
        ws_customers.cell(row=row_idx, column=col_idx, value=str(value)).alignment = Alignment(horizontal="center")

add_border(ws_customers, 28, 30, 1, len(response_summary.columns))

for col in ["A", "B", "C", "D", "E", "F", "G"]:
    ws_customers.column_dimensions[col].width = 22

print("Customer Summary sheet created.")

Customer Summary sheet created.


In [7]:
# CELL 6: CHARTS SHEET

ws_charts = wb.create_sheet("Charts")

# Title
ws_charts.merge_cells("A1:N1")
ws_charts["A1"].value = "Visual Reports"
style_title(ws_charts["A1"])
ws_charts.row_dimensions[1].height = 30

# List of charts to embed
charts = [
    ("monthly_sales_trend.png",       "A3",  "Monthly Sales Trend"),
    ("yearly_sales.png",              "J3",  "Yearly Sales"),
    ("top10_customers.png",           "A23", "Top 10 Customers"),
    ("response_rate.png",             "J23", "Promotion Response Rate"),
    ("transaction_distribution.png",  "A43", "Transaction Distribution"),
    ("customer_spend_distribution.png","A63","Customer Spend Distribution"),
]

for filename, cell, label in charts:
    full_path = CHART_PATH + filename
    try:
        img = XLImage(full_path)
        img.width  = 480
        img.height = 280
        ws_charts.add_image(img, cell)
        print(f"  Added chart : {label}")
    except Exception as e:
        print(f"  Could not add {filename}: {e}")

print("Charts sheet created.")

  Added chart : Monthly Sales Trend
  Added chart : Yearly Sales
  Added chart : Top 10 Customers
  Added chart : Promotion Response Rate
  Added chart : Transaction Distribution
  Added chart : Customer Spend Distribution
Charts sheet created.


In [8]:
# CELL 7: SAVE EXCEL FILE

OUTPUT_PATH = r"C:\Users\Asus\Downloads\archive\Retail_Sales_Report.xlsx"

wb.save(OUTPUT_PATH)
print("Excel report saved successfully.")
print(f"Location : {OUTPUT_PATH}")
print("")
print("Report contains the following sheets:")
for sheet in wb.sheetnames:
    print(f"  - {sheet}")

Excel report saved successfully.
Location : C:\Users\Asus\Downloads\archive\Retail_Sales_Report.xlsx

Report contains the following sheets:
  - Executive Summary
  - Monthly Sales
  - Customer Summary
  - Charts


In [10]:
# CELL 8: FIX DATE FORMAT IN CUSTOMER SUMMARY SHEET

ws_customers = wb["Customer Summary"]

# Fix date columns (columns E and F, rows 5 to 24)
for row in range(5, 25):
    for col in [5, 6]:
        cell = ws_customers.cell(row=row, column=col)
        if cell.value:
            # Remove timestamp, keep date only
            date_str = str(cell.value).split(" ")[0]
            cell.value = date_str
            cell.alignment = Alignment(horizontal="center")

wb.save(OUTPUT_PATH)
print("Date format fixed and file saved.")

Date format fixed and file saved.
